# Nettoyage et préparation des données – BAAC 2024

Ce notebook présente les différentes étapes de préparation des données issues de la **Base des Accidents Corporels de la Circulation (BAAC) 2024**. Il automatise le chargement, le nettoyage, le traitement des valeurs manquantes, la fusion des différents fichiers et la création d'un jeu de données consolidé, prêt pour l'analyse exploratoire et le Machine Learning.

## Objectifs

- Charger les quatre fichiers de la base BAAC 2024.
- Vérifier la qualité des données.
- Nettoyer les données (gestion des doublons, des incohérences et des valeurs manquantes).
- Fusionner les différentes tables à l'aide des clés communes.
- Créer un jeu de données consolidé et prêt pour l'analyse exploratoire des données (EDA) et la modélisation.
- Exporter le jeu de données nettoyé au format CSV.

Ce fichier servira de base aux étapes suivantes du projet :

- Analyse exploratoire des données (EDA)
- Feature Engineering
- Entraînement et évaluation des modèles de Machine Learning
- Création de cartes interactives avec Folium
- Développement du tableau de bord

In [1]:
import pandas as pd
import numpy as np
import os
import glob

## 1. Chargement des données

Nous configurons les répertoires et chargeons les fichiers csv en spécifiant le séparateur `;` et l'encodage `utf-8`.

In [3]:
raw_dir = os.path.join("data", "raw")
processed_dir = os.path.join("data", "processed")

caract_path = os.path.join(raw_dir, "Caract_2024.csv")
lieux_path = os.path.join(raw_dir, "Lieux_2024.csv")
vehicules_path = os.path.join(raw_dir, "Vehicules_2024.csv")
usagers_path = os.path.join(raw_dir, "Usagers_2024.csv")

# Chargement des fichiers
df_caract = pd.read_csv(caract_path, sep=';', encoding='utf-8', low_memory=False)
df_vehicules = pd.read_csv(vehicules_path, sep=';', encoding='utf-8', low_memory=False)
df_usagers = pd.read_csv(usagers_path, sep=';', encoding='utf-8', low_memory=False)
df_lieux= pd.read_csv(lieux_path, sep=';',encoding='utf-8',  low_memory=False)
print(f"Caractéristiques : {df_caract.shape[0]} lignes, {df_caract.shape[1]} colonnes")
print(f"Véhicules : {df_vehicules.shape[0]} lignes, {df_vehicules.shape[1]} colonnes")
print(f"Usagers : {df_usagers.shape[0]} lignes, {df_usagers.shape[1]} colonnes")
print(f"Lieux : {df_lieux.shape[0]} lignes, {df_lieux.shape[1]} colonnes")


Caractéristiques : 54402 lignes, 15 colonnes
Véhicules : 92678 lignes, 11 colonnes
Usagers : 125187 lignes, 16 colonnes
Lieux : 70248 lignes, 18 colonnes


## 2. Nettoyage des identifiants 

Les exports ONISR du fichier BAAC contiennent des **espaces**  dans les identifiants numériques (par exemple, `155 781 758` au lieu de `155781758`). Cela empêche les conversions de type et bloque les jointures.

Nous allons nettoyer ces identifiants en supprimant tous les espaces.

In [4]:
df_vehicules.head()

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155 781 758,A01,1,7,0,2,1,13,1,NaN
1,202400000001,155 781 759,B01,2,14,0,2,2,21,1,NaN
2,202400000002,155 781 757,A01,1,10,0,1,3,15,1,NaN
3,202400000003,155 781 756,A01,3,3,9,1,1,1,1,NaN
4,202400000004,155 781 754,B01,3,34,0,2,0,0,1,NaN


In [8]:
def clean_identifiers(df, col_name):
    if col_name in df.columns:
        # Remplacer les caractères d'espacement par une chaîne vide
        df[col_name] = df[col_name].astype(str).str.replace(r'\s+', '', regex=True)
        # Convertir les 'nan' textuels en vrais NaN
        df[col_name] = df[col_name].replace({'nan': np.nan, '': np.nan})
    return df

# Nettoyage des identifiants d'accidents et de véhicules
df_caract = clean_identifiers(df_caract, 'Num_Acc')

df_vehicules = clean_identifiers(df_vehicules, 'Num_Acc')
df_vehicules = clean_identifiers(df_vehicules, 'id_vehicule')

df_usagers = clean_identifiers(df_usagers, 'Num_Acc')
df_usagers = clean_identifiers(df_usagers, 'id_vehicule')
df_usagers = clean_identifiers(df_usagers, 'id_usager')
df_lieux = clean_identifiers(df_lieux, 'Num_Acc')

print("Identifiant de véhicule nettoyé dans Usagers  :", df_vehicules['id_vehicule'].iloc[0])

df_vehicules.head()

Identifiant de véhicule nettoyé dans Usagers  : 155781758


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155781758,A01,1,7,0,2,1,13,1,NaN
1,202400000001,155781759,B01,2,14,0,2,2,21,1,NaN
2,202400000002,155781757,A01,1,10,0,1,3,15,1,NaN
3,202400000003,155781756,A01,3,3,9,1,1,1,1,NaN
4,202400000004,155781754,B01,3,34,0,2,0,0,1,NaN


## 3. Standardisation des valeurs manquantes et espaces textuels

Les valeurs manquantes ou non renseignées dans le fichier BAAC peuvent prendre la forme de :
- Chaînes avec des espaces superflus (ex: `" -1"` ou `" . "`)
- Codes numériques de non-renseignement (ex: `-1` ou `0` pour certaines variables)

Nous allons donc :
1. Enlever les espaces au début/fin des colonnes textuelles.
2. Standardiser les `-1` et valeurs assimilées en vrais `NaN` dans Pandas.

In [9]:
# Nettoyage des espaces pour toutes les chaînes
dfs_list = [df_caract, df_vehicules, df_usagers,df_lieux] 
   
for df in dfs_list:
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()

# Standardisation des valeurs manquantes textuelles et numériques
missing_reprs = ['-1', '-1.0', '.', 'nan', '', 'None']

for df in dfs_list:
    df.replace(missing_reprs, np.nan, inplace=True)
    # Remplacer les -1 numériques
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(-1, np.nan)

print("Taux de valeurs manquantes par colonne dans la table Usagers (%):")
print((df_usagers.isnull().sum() / len(df_usagers) * 100).sort_values(ascending=False).head(10))

Taux de valeurs manquantes par colonne dans la table Usagers (%):
etatp      91.808255
secu3      90.371205
locp       49.342983
actp       49.307037
secu2      42.986093
trajet      2.097662
an_nais     2.060118
sexe        1.913138
secu1       1.679887
place       0.002396
dtype: float64


## 4. Spécificité 2024 : Véhicules sans Usagers (Délits de fuite)

Il arrive que des véhicules impliqués soient enregistrés dans `VEHICULES` mais que leur conducteur n'ait pas été identifié ou retrouvé. Ils ne figurent alors pas dans la table `USAGERS`.

In [10]:
veh_ids = set(df_vehicules['id_vehicule'].dropna())
usager_veh_ids = set(df_usagers['id_vehicule'].dropna())

missing_drivers = veh_ids - usager_veh_ids
print(f"Nombre total de véhicules sans usagers recensés : {len(missing_drivers)}")
print("Exemples d'id_vehicule en fuite :", list(missing_drivers)[:10])

Nombre total de véhicules sans usagers recensés : 24
Exemples d'id_vehicule en fuite : ['155719190', '155781755', '155776710', '155727799', '155708039', '155694791', '155731509', '155732108', '155698971', '155768083']


## 5. Jointure des tables

Nous fusionnons les tables pour obtenir une base globale au grain le plus fin (un usager impliqué par ligne) :
1. **`USAGERS`** comme table principale.
2. Jointure gauche avec **`VEHICULES`** sur `Num_Acc` et `id_vehicule`.
3. Jointure gauche avec **`CARACTERISTIQUES`** sur `Num_Acc`.
4. Jointure gauche avec **`LIEUX`** sur `Num_Acc` (si disponible).

In [11]:
# Fusion 1 : Usagers + Véhicules
df_merged = pd.merge(df_usagers, df_vehicules, on=['Num_Acc', 'id_vehicule'], how='left', suffixes=('_usager', '_vehicule'))

# Fusion 2 : + Caractéristiques
df_merged = pd.merge(df_merged, df_caract, on='Num_Acc', how='left')

# Fusion 3 : + Lieux 
if df_lieux is not None:
    df_merged = pd.merge(df_merged, df_lieux, on='Num_Acc', how='left')

print(f"Dimensions de la base finale fusionnée : {df_merged.shape[0]} lignes, {df_merged.shape[1]} colonnes")

Dimensions de la base finale fusionnée : 162548 lignes, 56 colonnes


## 6. Formatage des types et variables calculées

1. **GPS** : Conversion des latitudes/longitudes en nombres réels (remplacement de la virgule décimale par un point).
2. **Âge** : Calcul de l'âge à partir de l'année de naissance et l'année de l'accident.
3. **Heure** : Formatage de l'heure en format standard HH:MM.

In [14]:
# 1. Coordonnées GPS
if 'lat' in df_merged.columns and 'long' in df_merged.columns:
    df_merged['lat'] = df_merged['lat'].astype(str).str.replace(',', '.').astype(float)
    df_merged['long'] = df_merged['long'].astype(str).str.replace(',', '.').astype(float)

# 2. Calcul de l'âge
df_merged['an_nais'] = pd.to_numeric(df_merged['an_nais'], errors='coerce')
df_merged['an'] = pd.to_numeric(df_merged['an'], errors='coerce')
df_merged['age'] = df_merged['an'] - df_merged['an_nais']

# 3. Formatage de l'heure
def format_time(x):
    if pd.isnull(x):
        return np.nan
    x_str = str(x).strip().replace(':', '')
    if len(x_str) == 3:
        x_str = '0' + x_str
    if len(x_str) == 4:
        return f"{x_str[:2]}:{x_str[2:]}"
    return np.nan

df_merged['heure_formatee'] = df_merged['hrmn'].apply(format_time)

print("Aperçu des données calculées (les 5 premières lignes) :")
df_merged[['Num_Acc', 'age', 'sexe', 'grav', 'heure_formatee']].head()

Aperçu des données calculées (les 5 premières lignes) :


,Num_Acc,age,sexe,grav,heure_formatee
0,202400000001,21.0,1.0,3,07:40
1,202400000001,27.0,1.0,1,07:40
2,202400000002,97.0,2.0,3,15:05
3,202400000002,97.0,2.0,3,15:05
4,202400000002,37.0,1.0,1,15:05


## 7. Sauvegarde des jeux de données nettoyés

À cette étape, nous générons deux jeux de données distincts afin de répondre à deux objectifs différents :

### 1. `data/processed/baac_clean.csv`

Ce fichier contient l'ensemble des données nettoyées et fusionnées. Il est destiné aux analyses exploratoires (EDA), aux visualisations, aux requêtes SQL et aux cartes interactives.

### 2. `data/processed/baac_features.csv`

Ce fichier est une version allégée de la base de données, contenant uniquement les variables pertinentes pour l'entraînement des modèles de Machine Learning.

Avant sa sauvegarde, plusieurs variables sont créées afin de faciliter la modélisation.

#### Variable cible : `is_grave`

La variable **`grav`** du fichier BAAC est codée selon la nomenclature suivante :

| Code | Signification |
|------|---------------|
| 1 | Indemne |
| 2 | Tué |
| 3 | Blessé hospitalisé |
| 4 | Blessé léger |

> **Important :** ces valeurs sont des **codes** et **ne représentent pas un ordre de gravité croissant**. Par exemple, la valeur **4** correspond à un **blessé léger** et non au niveau de gravité le plus élevé.

Pour entraîner un modèle de **classification binaire**, cette variable est transformée en une nouvelle variable appelée **`is_grave`** :

- **1** : Accident grave (Tué ou Blessé hospitalisé)
- **0** : Accident non grave (Indemne ou Blessé léger)

Cette transformation permet au modèle de répondre à une question simple :

> **L'accident est-il grave ou non ?**

#### Variable `is_male`

La variable **`sexe`** est également transformée en variable binaire :

- **1** : Homme
- **0** : Femme

Cette représentation facilite le traitement des données par les algorithmes de Machine Learning.

In [17]:
os.makedirs(processed_dir, exist_ok=True)

# 1. Sauvegarde complète
clean_path = os.path.join(processed_dir, "baac_clean.csv")
df_merged.to_csv(clean_path, index=False, sep=';', encoding='utf-8')
print(f"Base complète propre sauvegardée : {clean_path}")

# 2. Feature Engineering & Préparation Modèle
df_features = df_merged.copy()

# Variable Cible Binaire (1: Accident Grave (Tué ou Hospitalisé), 0: Indemne ou Léger)
df_features['grav_num'] = pd.to_numeric(df_features['grav'], errors='coerce')
df_features['is_grave'] = df_features['grav_num'].apply(lambda x: 1 if x in [2, 3] else (0 if x in [1, 4] else np.nan))

# Sexe binarisé (1: Homme, 0: Femme)
df_features['sexe_num'] = pd.to_numeric(df_features['sexe'], errors='coerce')
df_features['is_male'] = df_features['sexe_num'].apply(lambda x: 1 if x == 1 else (0 if x == 2 else np.nan))

# Sélection des colonnes pertinentes pour la modélisation
selected_cols = [
    'Num_Acc', 'id_usager', 'id_vehicule',
    'age', 'is_male', 'catu', 'is_grave',
    'catv', 'motor', 'obs', 'obsm',
    'lum', 'atm', 'col', 'agg', 'int'
]


lieux_cols = ['catr', 'circ', 'surf', 'vma']
for col in lieux_cols:
    if col in df_features.columns:
        selected_cols.append(col)

# Suppression des lignes avec gravité inconnue
df_features = df_features[selected_cols].dropna(subset=['is_grave'])

features_path = os.path.join(processed_dir, "baac_features.csv")
df_features.to_csv(features_path, index=False, sep=';', encoding='utf-8')

Base complète propre sauvegardée : data\processed\baac_clean.csv
Base allégée pour Machine Learning sauvegardée : data\processed\baac_features.csv
Nombre total de lignes pour entraînement : 162548
